# Biohub Cell Tracking: 0.946 LB

This update continues the progression from **0.934 → 0.939 → 0.941 → 0.946** Public LB, a total of **+0.012** from the original 0.934 solution. The core architecture is unchanged throughout — same UNet3D + node-transformer detection, dual-seed ensemble, ILP-based global linking, and bidirectional association framework. The **0.934 → 0.939** step widened safe division recovery while making bidirectional association more conservative. The **0.939 → 0.941** step redistributed the repair budget by widening the safe-division parent search while tightening its learned confirmation and gap closing. This **0.941 → 0.946** step keeps those repair settings untouched and extends the existing eight-view spatial TTA from detection outputs into the UNet features used for association.

| Component | 0.934 | 0.939 | 0.941 | 0.946 |
| --- | ---: | ---: | ---: | ---: |
| Safe division parent max distance | 7.0 µm | 7.0 µm | 9.0 µm | 9.0 µm |
| Safe division sister max distance | 12.0 µm | 14.0 µm | 14.0 µm | 14.0 µm |
| Sister symmetry gate | off | 0.6 | 0.6 | 0.6 |
| DeepCenter safe-division veto | off | on | on | on |
| DeepCenter safe-division threshold | 0.12 | 0.12 | 0.25 | 0.25 |
| Gap-close base radius | 5.8 µm | 5.8 µm | 5.0 µm | 5.0 µm |
| Bidirectional association weight | 0.30 | 0.15 | 0.15 | 0.15 |
| Bidirectional winner-agreement guard | on | removed | removed | removed |
| DeepCenter expected epoch | 500 | 2 | 2 | 2 |
| Detection spatial TTA views | 8 | 8 | 8 | 8 |
| Edge-feature spatial TTA | off | off | off | on |

**Recap: wider division search with a symmetry and DeepCenter guard (0.934 → 0.939).** The sister-distance limit widened from 12 µm to 14 µm to recover genuine divisions where the two daughters had drifted slightly farther apart. Since a wider radius alone also admits more spurious pairings, two safeguards compensate: a symmetry gate rejecting daughter candidates sitting at very uneven distances from the parent,

$$
\frac{|d_1 - d_2|}{(d_1 + d_2)/2} \le 0.6
$$

and the DeepCenter veto, so every candidate the wider geometric search proposes still needs independent confirmation before being accepted.

**Recap: a more conservative bidirectional fusion (0.934 → 0.939).** Association is normally scored only forward in time — for a cell at frame $t$, which detection at $t+1$ it most likely became. This pipeline also scores it in reverse — given a detection at $t+1$, which detection at $t$ it most likely came from. Let $p_f$ be the forward probability and $p_r$ the reverse probability for a candidate pair. The reverse weight dropped from 0.30 to 0.15, so $p_r$ acts as a lighter supporting vote rather than pulling $p_f$ as strongly, and the previous requirement that forward and reverse agree on the exact same winner (the agreement guard) was replaced with their harmonic mean directly:

$$
p_{\text{harmonic}} = \frac{2 \, p_f \, p_r}{p_f + p_r}
$$

which keeps the mutual-support behavior — a link needs support from both directions to score highly — without the hard all-or-nothing condition.

**Recap: redistributing the repair budget (0.939 → 0.941).** The association setup that reached 0.939 is left untouched; three repair-stage parameters are retuned instead. The safe-division **parent** radius increases from 7 µm to 9 µm. This is a different axis from the earlier 12 → 14 µm change — that one widened the allowed distance *between the two daughters*; this one widens the distance *from the parent to the missing daughter candidate*, recovering parent-child links the previous radius was cutting off. That wider parent search creates more room for false division proposals, so the DeepCenter confirmation threshold is raised from 0.12 to 0.25. Candidates admitted by the larger radius still have to clear a stricter learned confirmation before being recovered as divisions — the search widens, but the bar to pass it goes up with it. At the same time, gap closing is tightened, with its base radius reduced from 5.8 µm to 5.0 µm — a separate repair path that reconnects broken tracks rather than recovering divisions. Candidates for that repair now have to be spatially closer. Together these three changes shift where the repair budget is spent: more room is given specifically to plausible divisions, while a stricter DeepCenter threshold and a tighter gap-close radius hold precision elsewhere.

**What's new in 0.946: extending spatial TTA into association features.** The full 0.941 tracking and repair configuration is retained unchanged. Importantly, the improvement does **not** come from increasing the number of geometric TTA views. The 0.941 pipeline already evaluates detection using the full eight spatial transformations of the D4 symmetry group. The change in 0.946 is where the information from those eight views is used. The eight spatial views are:

| # | D4 Spatial View | Operation |
| ---: | --- | --- |
| 1 | Identity | Original orientation |
| 2 | Horizontal flip | Reflection across one spatial axis |
| 3 | Vertical flip | Reflection across the other spatial axis |
| 4 | Horizontal + vertical flip | Reflection across both axes, equivalent to a 180° rotation |
| 5 | Rotation 90° | 90° spatial rotation |
| 6 | Rotation 270° | 270° spatial rotation |
| 7 | Transpose | Reflection across one diagonal |
| 8 | Anti-transpose | Reflection across the opposite diagonal |

These eight operations cover the unique symmetries of a square under the D4 group. A separate 180° rotation does not need to be added because flipping both spatial axes already produces the same geometric transformation. Therefore, moving from 0.941 to 0.946 is not an **8-view → 16-view** expansion or a **4-view → 8-view** expansion. Both versions use the same eight geometric views. For an input $x$, let $T_i$ denote the $i$-th D4 transformation. Each transformed input $T_i(x)$ is passed through the detector, producing a detection-logit map. Because each output is expressed in the coordinate system of its transformed input, it first has to be mapped back to the original orientation using the corresponding inverse transformation $T_i^{-1}$. Detection TTA is therefore:

$$
\bar{L}(x) = \frac{1}{8}\sum_{i=1}^{8} T_i^{-1}\left(L(T_i(x))\right)
$$

where $L(T_i(x))$ denotes the detection logits produced from the $i$-th transformed view. This averaging reduces orientation-specific variation in the final detection map. That mechanism was **already present before 0.946**. In 0.941, all eight views contributed to $\bar{L}(x)$, so cell detection was already spatially ensembled. However, the UNet feature tensor used downstream for association did not receive the same treatment. Once detection TTA was complete, the edge predictor still relied on the UNet representation extracted from the original orientation. Conceptually, the 0.941 inference path was:

$$
\{T_i(x)\}_{i=1}^{8}
\rightarrow
\bar{L}(x)
\rightarrow
\text{detection}
$$

while the association representation remained:

$$
F_{\text{association}}(x) = F(x)
$$

where $F(x)$ is the UNet feature map from the original view only. This creates an asymmetry in the inference pipeline. Detection benefits from eight spatial observations, but the representation used to decide which detections should be connected through time still depends on a single orientation. The 0.946 update closes this mismatch by extending the same transform-invert-average procedure to the UNet feature maps. For every transformed input, the corresponding feature representation $F(T_i(x))$ is retained. Each feature map is then mapped back into the original coordinate system before averaging:

$$
\bar{F}(x) = \frac{1}{8}\sum_{i=1}^{8} T_i^{-1}\left(F(T_i(x))\right)
$$

The inverse transformation is important. A feature map extracted from a 90°-rotated input cannot be averaged directly with the original feature map because their spatial coordinates do not correspond. Applying $T_i^{-1}$ first realigns every representation to the same coordinate system. The edge predictor then receives:

$$
F_{\text{association}}(x) = \bar{F}(x)
$$

instead of:

$$
F_{\text{association}}(x) = F(x)
$$

so the difference between 0.941 and 0.946 can be summarized directly:

| TTA Component | 0.941 Public LB | 0.946 Public LB |
| --- | --- | --- |
| Number of spatial views | 8 D4 views | 8 D4 views |
| Identity | on | on |
| Horizontal flip | on | on |
| Vertical flip | on | on |
| Horizontal + vertical flip | on | on |
| Rotation 90° | on | on |
| Rotation 270° | on | on |
| Transpose | on | on |
| Anti-transpose | on | on |
| Detection-logit TTA | 8-view inverse-transform average | 8-view inverse-transform average |
| Detection representation | spatially ensembled | spatially ensembled |
| UNet edge-feature TTA | off | on |
| UNet features used by edge predictor | original view only | all 8 D4 views |
| Feature inverse transform | not used for edge-feature averaging | applied to each transformed feature map |
| Edge predictor input | $F(x)$ | $\bar{F}(x)$ |
| Association representation | single-view | spatially ensembled |
| Repair configuration | 0.941 settings | unchanged from 0.941 |

The key distinction is therefore not the transformations themselves, but **how far their information propagates through inference**. In **0.941**, the eight D4 views terminate at the detection ensemble:

$$
\text{8 D4 views}
\rightarrow
\text{detection logits}
\rightarrow
\text{inverse transform}
\rightarrow
\text{average}
\rightarrow
\text{detection}
$$

while association returns to a single-view representation:

$$
\text{original view}
\rightarrow
F(x)
\rightarrow
\text{edge predictor}
\rightarrow
\text{association}
$$

In **0.946**, the same eight views contribute to both paths:

$$
\text{8 D4 views}
\rightarrow
\begin{cases}
\text{detection logits}
\rightarrow
\text{inverse transform}
\rightarrow
\bar{L}(x)
\\
\text{UNet features}
\rightarrow
\text{inverse transform}
\rightarrow
\bar{F}(x)
\end{cases}
$$

and the averaged feature representation $\bar{F}(x)$ is then consumed by the edge predictor. This matters because association is not determined only by the detected cell coordinates. The edge predictor also uses learned UNet representations to evaluate candidate temporal links. Under the previous setup, an otherwise valid association could still be affected by orientation-specific variation in the original-view feature map even though detection itself had already been stabilized with TTA. Averaging the aligned feature maps reduces that dependence on a particular spatial orientation and makes the representation supplied to association consistent with the ensemble principle already used for detection. The change is deliberately narrow. It does not globally relax edge thresholds, increase repair radii, add more geometric transformations, or rewrite the linking procedure. The 9 µm safe-division parent radius, 14 µm sister radius, 0.6 symmetry gate, 0.25 DeepCenter safe-division threshold, 5.0 µm gap-close radius, 0.15 bidirectional association weight, harmonic forward-reverse fusion, dual-seed ensemble, ILP-based global linking, and the rest of the 0.941 repair configuration are retained. The **0.941 → 0.946** step instead makes better use of the eight spatial observations that were already being computed by extending their contribution from the detection output to the learned features used for temporal association.

**Summary.** The progression remains cumulative rather than a redesign. **0.934 → 0.939** widened safe division recovery with additional geometric and DeepCenter safeguards while making bidirectional association more conservative. **0.939 → 0.941** redistributed the repair budget by widening the parent-side division search, strengthening its DeepCenter confirmation, and tightening gap closing. **0.941 → 0.946** leaves those tracking and repair settings unchanged and extends the existing eight-view D4 ensemble from detection logits into the UNet features consumed by the edge predictor. The number and type of spatial transforms do not change; what changes is that association now benefits from the same spatially ensembled representation that detection already had. The resulting progression is **0.934 → 0.939 → 0.941 → 0.946**, for a cumulative **+0.012 Public LB** improvement over the original 0.934 solution and a **+0.005** improvement over the previous 0.941 configuration.

**If you have ideas on how to push this toward 0.950, feel free to reach out — I'm also looking for a teammate for this competition. Also, if you find this useful, an upvote would be appreciated!**

In [ ]:
from __future__ import annotations
# --- P100 escape hatch (ours) -------------------------------------------------
# Installs torch 2.5.1+cu121 from a mounted wheelhouse when this session drew a
# Tesla P100 (sm_60), which the image torch cannot run. No-op on a T4.
import subprocess as _sp, sys as _sys, pathlib as _pl
# Found by globbing, not by a hardcoded mount path. v1 hardcoded
# /kaggle/input/claude-torch-wheelhouse/wheels, the directory was not there, and the
# `is_dir()` guard turned that into a silent skip -- MEMORY.md's recurring bug class,
# silent-pass-on-missing-input, reproduced exactly. Now a P100 with no wheel is loud.
_wheels = next((p.parent for p in
                _pl.Path("/kaggle/input").glob("*/**/torch-*.whl")), None)
try:
    _name = _sp.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                    capture_output=True, text=True, timeout=60).stdout
except Exception:
    _name = ""
if "P100" not in _name:
    print(f"GPU {_name.strip()!r} -- no torch replacement needed", flush=True)
elif _wheels is None:
    print("P100 AND NO WHEELHOUSE -- this run will die in the first forward pass.",
          flush=True)
    for _p in sorted(_pl.Path("/kaggle/input").glob("*")):
        print("   mounted:", _p.name, flush=True)
else:
    print(f"P100 detected -- installing torch 2.5.1+cu121 from {_wheels}", flush=True)
    # No --no-deps: torch 2.5.1 needs the cu121 nvidia-* runtimes, and the image ships
    # cu128 ones. The wheelhouse carries the full closure (cublas, cudnn, nccl, triton,
    # sympy ...), so let pip resolve it there. torchvision is NOT in the wheelhouse and
    # is not requested -- the fork's detector is a custom UNet3D that does not import it.
    _r = _sp.run(["/usr/bin/python3", "-m", "pip", "install", "--no-index",
                  "--find-links", str(_wheels), "--force-reinstall", "torch==2.5.1+cu121"],
                 capture_output=True, text=True, timeout=3600)
    print(f"wheelhouse install rc={_r.returncode}", flush=True)
    if _r.returncode:
        print(_r.stdout[-2000:], _r.stderr[-2000:], flush=True)
# ------------------------------------------------------------------------------
import subprocess as _gsp
try:
    _gpu = _gsp.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                    capture_output=True, text=True, timeout=60).stdout.strip()
except Exception:
    _gpu = ""
print("accelerator:", _gpu, flush=True)
if "P100" in _gpu:
    print("P100 -- temporal attention will be chunked to stay inside the sm_60 "
          "batched-GEMM limit", flush=True)
import os

os.environ['BIOHUB_MODEL_ARTIFACTS'] = '/kaggle/input/datasets/reyhanksatria/biohub-tracking-support-pack'
os.environ['BIOHUB_TARGET_ARTIFACT_SLUG'] = 'biohub-tracking-support-pack'
os.environ['BIOHUB_ALLOW_ARTIFACT_FALLBACK'] = '1'
os.environ['BIOHUB_DEEPCENTER_CHECKPOINT'] = '/kaggle/input/datasets/reyhanksatria/biohub-deepcenterunet3d-center-prior-v1/weights/full_frame_center/best.pt'
os.environ['BIOHUB_SECONDARY_ARTIFACT_MANIFEST'] = '/kaggle/input/datasets/reyhanksatria/biohub-temporalunet3d-seed-314159-v1/ARTIFACT_MANIFEST.json'

# Decode hexadecimal text into the exact UTF-8 source string
def _exact_text(hex_text):
    return bytes.fromhex(hex_text).decode('utf-8')

# Normalize multiline patch text with controlled indentation and trailing newlines
def _patch_text(text, indent = 0, trailing_newline = False):
    prefix = ' ' * indent
    lines = text.splitlines()

    if lines and lines[0].strip().startswith('# '):
        lines = lines[1:]

    if lines and (not lines[0].strip()):
        lines = lines[1:]

    if lines and (not lines[-1].strip()):
        lines = lines[:-1]
    value = chr(10).join((prefix + line if line else '' for line in lines))
    return value + (chr(10) if trailing_newline else '')

import os
from collections import Counter

# Configure the verified production lineage and retain the 0.941 repair settings
BIOHUB_PRESET = 'harmonic_v3_division_wide'
BIOHUB_SCORE_AXIS = '0.933 baseline -> 0.934 harmonic fusion -> 0.939 wider divisions/calmer fusion -> 0.941 repair adaptation -> 0.946 edge-feature TTA'
os.environ['BIOHUB_OUTPUT_FILTER_SHORT_TRACKS'] = '1'
os.environ['BIOHUB_DET_THRESHOLD'] = '0.965'
os.environ['BIOHUB_MOTION_RELINK_LEARNED_BONUS'] = '1.0'
os.environ['BIOHUB_ILP_APPEARANCE_WEIGHT'] = '0.0'
os.environ['BIOHUB_ILP_DISAPPEARANCE_WEIGHT'] = '2'
os.environ['BIOHUB_GAP_CLOSE_MAX_GAP'] = '2'
os.environ['BIOHUB_GAP_CLOSE_UM'] = '5.0'
os.environ['BIOHUB_GAP_DENSITY_ADAPTIVE'] = '1'
os.environ['BIOHUB_GAP_DENSITY_REFERENCE_UM'] = '6.5'
os.environ['BIOHUB_GAP_DENSITY_GAIN'] = '0.040'
os.environ['BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM'] = '0.125'
os.environ['BIOHUB_GAP_DENSITY_NEIGHBORS'] = '3'
os.environ['BIOHUB_OUTPUT_MIN_TRACK_LEN'] = '6'
os.environ['BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS'] = '1'
os.environ['BIOHUB_OUTPUT_GAP2_RECOVERY'] = '1'
os.environ['BIOHUB_SAFE_DIV_MAX_UM'] = '9.0'
os.environ['BIOHUB_SAFE_DIV_SISTER_MAX_UM'] = '14.0'
os.environ['BIOHUB_SAFE_DIV_SISTER_SYMMETRY_TAU'] = '0.6'
os.environ['BIOHUB_SAFE_DIV_DIVERGE_UM'] = '2.25'
os.environ['BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM'] = '10.0'
os.environ['BIOHUB_SAFE_DIV_FRAME_FRAC_CAP'] = '0.0076'
os.environ['BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP'] = '0.00375'
os.environ['BIOHUB_ILP_DIVISION_WEIGHT'] = '1.2'
os.environ['BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE'] = '1'
os.environ['BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN'] = '4'
os.environ['BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB'] = '0.88'
os.environ['BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM'] = '3.0'
os.environ['BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC'] = '0.012'
os.environ['BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS'] = '120'
os.environ['BIOHUB_USE_DEEPCENTER_VETO'] = '1'
os.environ['BIOHUB_REQUIRE_DEEPCENTER_VETO'] = '1'
os.environ['BIOHUB_DEEPCENTER_EXPECTED_EPOCH'] = '2'
os.environ['BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM'] = '8.5'
os.environ['BIOHUB_DEEPCENTER_CHECKPOINT'] = '/kaggle/input/datasets/reyhanksatria/biohub-deepcenterunet3d-center-prior-v1/weights/full_frame_center/best.pt'
os.environ['BIOHUB_DEEPCENTER_GAP_VETO'] = '1'
os.environ['BIOHUB_DEEPCENTER_GAP_THRESHOLD'] = '0.25'
os.environ['BIOHUB_DEEPCENTER_SAFE_DIV_VETO'] = '1'
os.environ['BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD'] = '0.25'
os.environ['BIOHUB_RUN_OUTPUT_DIAGNOSTICS'] = '0'
os.environ['BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT'] = '0.15'
os.environ['BIOHUB_BIDIRECTIONAL_FUSION_MODE'] = 'harmonic_probability'
os.environ['BIOHUB_EDGE_FEATURE_TTA'] = '1'
os.environ['BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION'] = '0.90'
os.environ['BIOHUB_DIAGNOSTIC_ARM'] = 'harmonic_association_production'
print('BIOHUB_PRESET:', BIOHUB_PRESET)
print('BIOHUB_SCORE_AXIS:', BIOHUB_SCORE_AXIS)

import json as _guard_json
import math as _guard_math
import os as _guard_os

# Validate critical environment settings before starting the pipeline
_EXPECTED_NUMERIC = {'BIOHUB_DET_THRESHOLD': 0.965, 'BIOHUB_ILP_APPEARANCE_WEIGHT': 0.0, 'BIOHUB_ILP_DISAPPEARANCE_WEIGHT': 2, 'BIOHUB_GAP_CLOSE_UM': 5.0, 'BIOHUB_OUTPUT_MIN_TRACK_LEN': 6.0, 'BIOHUB_SAFE_DIV_MAX_UM': 9.0, 'BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD': 0.25, 'BIOHUB_BIDIRECTIONAL_EDGE_WEIGHT': 0.15}
_EXPECTED_TEXT = {'BIOHUB_BIDIRECTIONAL_FUSION_MODE': 'harmonic_probability', 'BIOHUB_DUAL_SEED_MIN_CANDIDATE_RETENTION': '0.90'}
_drift = {}

for _key, _want in _EXPECTED_NUMERIC.items():
    _raw = _guard_os.environ.get(_key)

    if _raw is None:
        _drift[_key] = 'missing'
        continue
    _got = float(_raw)

    if not _guard_math.isclose(_got, _want, rel_tol = 0.0, abs_tol = 1e-12):
        _drift[_key] = {'expected': _want, 'actual': _got}

for _key, _want in _EXPECTED_TEXT.items():
    _got = _guard_os.environ.get(_key)

    if _got != _want:
        _drift[_key] = {'expected': _want, 'actual': _got}

if _drift:
    raise RuntimeError('Configuration drift detected: ' + _guard_json.dumps(_drift, sort_keys = True))

print('Configuration guard: PASS')
print('Verified score progression: 0.933 -> 0.934 -> 0.939 -> 0.941 -> 0.946')
print('Previous verified configuration: public LB 0.941')
print('Current verified configuration: public LB 0.946')
print('0.941 repair settings retained: parent radius 9.0, DeepCenter threshold 0.25, gap-close radius 5.0')
print('0.946 change: eight-view D4 averaging now also covers the UNet features used by association')
print('Reverse-time association weight retained at 0.150')

import csv
import importlib.util
import json
import math
import os
import shutil
import subprocess
import tempfile
import zipfile
import sys
import time
from pathlib import Path
import pandas as pd
from IPython.display import display

# Resolve Kaggle competition paths and initialize output locations
COMPETITION = 'biohub-cell-tracking-during-development'
COMP_DIR_CANDIDATES = [Path(f'/kaggle/input/competitions/{COMPETITION}'), Path(f'/kaggle/input/{COMPETITION}')]
COMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])
TEST_DIR = COMP_DIR / 'test'
WORKING_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
REPO_DIR = WORKING_DIR / 'tracking_repo'
SUBMISSION_PATH = WORKING_DIR / 'submission.csv'
RUN_STATS_PATH = WORKING_DIR / 'run_stats.csv'
METHOD = 'unet_transformer'
WEIGHTS_RELATIVE = f'weights/{METHOD}/split_0/edge_predictor_best.pth'
EXPERIMENT_TAG = 'edge_feature_tta_0946'
TARGET_ARTIFACT_SLUG = os.environ.get('BIOHUB_TARGET_ARTIFACT_SLUG', 'biohub-tracking-support-pack-50ep-v1')
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get('BIOHUB_PRIMARY_ARTIFACT_MANIFEST', '/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json'))
ALLOW_ARTIFACT_FALLBACK = os.environ.get('BIOHUB_ALLOW_ARTIFACT_FALLBACK', '0') != '0'
DET_THRESHOLD = float(os.environ.get('BIOHUB_DET_THRESHOLD', '0.99'))
UNET_BATCH_SIZE = int(os.environ.get('BIOHUB_UNET_BATCH_SIZE', '4'))
USE_ILP = os.environ.get('BIOHUB_USE_ILP', '1') != '0'
ILP_EDGE_WEIGHT = float(os.environ.get('BIOHUB_ILP_EDGE_WEIGHT', '-1.0'))
ILP_APPEARANCE_WEIGHT = float(os.environ.get('BIOHUB_ILP_APPEARANCE_WEIGHT', '0.1'))
ILP_DISAPPEARANCE_WEIGHT = float(os.environ.get('BIOHUB_ILP_DISAPPEARANCE_WEIGHT', '0.1'))
ILP_DIVISION_WEIGHT = float(os.environ.get('BIOHUB_ILP_DIVISION_WEIGHT', '1.0'))
SLICE = ''
ALLOW_PIP_INSTALL = os.environ.get('BIOHUB_ALLOW_PIP_INSTALL', '0') != '0'
RUN_OUTPUT_DIAGNOSTICS = os.environ.get('BIOHUB_RUN_OUTPUT_DIAGNOSTICS', '1') != '0'
OUTPUT_EDGE_MAX_UM = float(os.environ.get('BIOHUB_OUTPUT_EDGE_MAX_UM', '14.0'))
OUTPUT_ENFORCE_NEXT_FRAME = os.environ.get('BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME', '1') != '0'
OUTPUT_SINGLE_PARENT_REPAIR = os.environ.get('BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR', '1') != '0'
OUTPUT_SINGLE_CHILD_REPAIR = os.environ.get('BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR', '0') != '0'
OUTPUT_PRUNE_ISOLATED = os.environ.get('BIOHUB_OUTPUT_PRUNE_ISOLATED', '1') != '0'
OUTPUT_MOTION_RELINK = os.environ.get('BIOHUB_OUTPUT_MOTION_RELINK', '1') != '0'
MOTION_RELINK_TIGHT_UM = float(os.environ.get('BIOHUB_MOTION_RELINK_TIGHT_UM', '6.0'))
MOTION_RELINK_RELAXED_UM = float(os.environ.get('BIOHUB_MOTION_RELINK_RELAXED_UM', '10.0'))
MOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get('BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT', '0.5'))
MOTION_RELINK_LEARNED_BONUS = float(os.environ.get('BIOHUB_MOTION_RELINK_LEARNED_BONUS', '0.75'))
MOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get('BIOHUB_MOTION_RELINK_MAX_FRAME_NODES', '2600'))
OUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get('BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER', '0') != '0'
DIV_PARENT_MAX_UM = float(os.environ.get('BIOHUB_DIV_PARENT_MAX_UM', '10.5'))
DIV_SISTER_MAX_UM = float(os.environ.get('BIOHUB_DIV_SISTER_MAX_UM', '8.0'))
DIV_DROP_TO_SINGLE_IF_BAD = os.environ.get('BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD', '1') != '0'
OUTPUT_GAP_CLOSE = os.environ.get('BIOHUB_OUTPUT_GAP_CLOSE', '1') != '0'
GAP_CLOSE_MAX_GAP = int(os.environ.get('BIOHUB_GAP_CLOSE_MAX_GAP', '1'))
GAP_CLOSE_UM = float(os.environ.get('BIOHUB_GAP_CLOSE_UM', '6.0'))
GAP_DENSITY_ADAPTIVE = os.environ.get('BIOHUB_GAP_DENSITY_ADAPTIVE', '0') != '0'
GAP_DENSITY_REFERENCE_UM = float(os.environ.get('BIOHUB_GAP_DENSITY_REFERENCE_UM', '6.5'))
GAP_DENSITY_GAIN = float(os.environ.get('BIOHUB_GAP_DENSITY_GAIN', '0.040'))
GAP_DENSITY_MAX_STEP_DELTA_UM = float(os.environ.get('BIOHUB_GAP_DENSITY_MAX_STEP_DELTA_UM', '0.125'))
GAP_DENSITY_NEIGHBORS = int(os.environ.get('BIOHUB_GAP_DENSITY_NEIGHBORS', '3'))
GAP_CLOSE_REUSE_EXISTING = os.environ.get('BIOHUB_GAP_CLOSE_REUSE_EXISTING', '1') != '0'
GAP_CLOSE_REUSE_UM = float(os.environ.get('BIOHUB_GAP_CLOSE_REUSE_UM', '3.2'))
GAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get('BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC', '0.05'))
GAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get('BIOHUB_GAP_CLOSE_MAX_ADDED_ABS', '2000'))
GAP_REFINE_SYNTHETIC = os.environ.get('BIOHUB_GAP_REFINE_SYNTHETIC', '1') != '0'
GAP_REFINE_WIN_Z = int(os.environ.get('BIOHUB_GAP_REFINE_WIN_Z', '1'))
GAP_REFINE_WIN_YX = int(os.environ.get('BIOHUB_GAP_REFINE_WIN_YX', '3'))
GAP_REFINE_MAX_SHIFT_UM = float(os.environ.get('BIOHUB_GAP_REFINE_MAX_SHIFT_UM', '3.2'))
OUTPUT_FILTER_SHORT_TRACKS = os.environ.get('BIOHUB_OUTPUT_FILTER_SHORT_TRACKS', '1') != '0'
OUTPUT_MIN_TRACK_LEN = int(os.environ.get('BIOHUB_OUTPUT_MIN_TRACK_LEN', '6'))
OUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get('BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS', '1') != '0'
ADAPTIVE_SHORT_TRACK_RESCUE = os.environ.get('BIOHUB_ADAPTIVE_SHORT_TRACK_RESCUE', '0') != '0'
SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC = float(os.environ.get('BIOHUB_SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC', '0.10'))
SHORT_TRACK_RESCUE_MIN_LEN = int(os.environ.get('BIOHUB_SHORT_TRACK_RESCUE_MIN_LEN', '4'))
SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB = float(os.environ.get('BIOHUB_SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB', '0.82'))
SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM = float(os.environ.get('BIOHUB_SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM', '3.25'))
SHORT_TRACK_RESCUE_MAX_NODES_FRAC = float(os.environ.get('BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_FRAC', '0.018'))
SHORT_TRACK_RESCUE_MAX_NODES_ABS = int(os.environ.get('BIOHUB_SHORT_TRACK_RESCUE_MAX_NODES_ABS', '180'))
OUTPUT_LINEFIT_SMOOTH = os.environ.get('BIOHUB_OUTPUT_LINEFIT_SMOOTH', '1') != '0'
OUTPUT_LINEFIT_WEIGHT = float(os.environ.get('BIOHUB_OUTPUT_LINEFIT_WEIGHT', '0.8'))
OUTPUT_LINEFIT_WINDOW = int(os.environ.get('BIOHUB_OUTPUT_LINEFIT_WINDOW', '2'))
OUTPUT_GAP2_RECOVERY = os.environ.get('BIOHUB_OUTPUT_GAP2_RECOVERY', '0') != '0'
GAP2_MAX_TOTAL_UM = float(os.environ.get('BIOHUB_GAP2_MAX_TOTAL_UM', '10.2'))
GAP2_MAX_STEP_UM = float(os.environ.get('BIOHUB_GAP2_MAX_STEP_UM', '4.4'))
GAP2_MAX_LINKS_FRAC = float(os.environ.get('BIOHUB_GAP2_MAX_LINKS_FRAC', '0.0045'))
GAP2_MAX_LINKS_ABS = int(os.environ.get('BIOHUB_GAP2_MAX_LINKS_ABS', '180'))
GAP2_REQUIRE_CONTEXT = os.environ.get('BIOHUB_GAP2_REQUIRE_CONTEXT', '1') != '0'
GAP2_FRAME_FRAC_CAP = float(os.environ.get('BIOHUB_GAP2_FRAME_FRAC_CAP', '0.006'))
OUTPUT_SAFE_DIVISIONS = os.environ.get('BIOHUB_OUTPUT_SAFE_DIVISIONS', '1') != '0'
SAFE_DIV_MAX_UM = float(os.environ.get('BIOHUB_SAFE_DIV_MAX_UM', '4.7'))
SAFE_DIV_SISTER_MAX_UM = float(os.environ.get('BIOHUB_SAFE_DIV_SISTER_MAX_UM', '7.2'))
SAFE_DIV_SISTER_SYMMETRY_TAU = float(os.environ.get('BIOHUB_SAFE_DIV_SISTER_SYMMETRY_TAU', '0.0'))
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get('BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM', '7.8'))
SAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get('BIOHUB_SAFE_DIV_FRAME_FRAC_CAP', '0.008'))
SAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get('BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP', '0.004'))
SAFE_DIV_DIVERGE_UM = float(os.environ.get('BIOHUB_SAFE_DIV_DIVERGE_UM', '2.25'))
SAFE_DIV_REQUIRE_DIVERGENCE = os.environ.get('BIOHUB_SAFE_DIV_REQUIRE_DIVERGENCE', '1') != '0'
SAFE_DIV_REQUIRE_MUTUAL_NN = os.environ.get('BIOHUB_SAFE_DIV_REQUIRE_MUTUAL_NN', '1') != '0'
USE_DEEPCENTER_VETO = os.environ.get('BIOHUB_USE_DEEPCENTER_VETO', '1') != '0'
REQUIRE_DEEPCENTER_VETO = os.environ.get('BIOHUB_REQUIRE_DEEPCENTER_VETO', '1') != '0'
DEEPCENTER_MANIFEST_DEFAULT = os.environ.get('BIOHUB_DEEPCENTER_MANIFEST_DEFAULT', '/kaggle/input/datasets/pilkwang/biohub-deepcenter-unet3d-center-prior-v1/ARTIFACT_MANIFEST.json')
DEEPCENTER_CHECKPOINT_DEFAULT = os.environ.get('BIOHUB_DEEPCENTER_CHECKPOINT_DEFAULT', '/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/best.pt')
DEEPCENTER_RELATIVE = os.environ.get('BIOHUB_DEEPCENTER_RELATIVE', 'weights/full_frame_center/best.pt')
DEEPCENTER_GAP_VETO = os.environ.get('BIOHUB_DEEPCENTER_GAP_VETO', '1') != '0'
DEEPCENTER_SAFE_DIV_VETO = os.environ.get('BIOHUB_DEEPCENTER_SAFE_DIV_VETO', '1') != '0'
DEEPCENTER_GAP_THRESHOLD = float(os.environ.get('BIOHUB_DEEPCENTER_GAP_THRESHOLD', '0.10'))
DEEPCENTER_EXPECTED_EPOCH = int(os.environ.get('BIOHUB_DEEPCENTER_EXPECTED_EPOCH', '0'))
DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM = float(os.environ.get('BIOHUB_DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM', '0'))
DEEPCENTER_SAFE_DIV_THRESHOLD = float(os.environ.get('BIOHUB_DEEPCENTER_SAFE_DIV_THRESHOLD', '0.12'))
DEEPCENTER_SCORE_WIN_Z = int(os.environ.get('BIOHUB_DEEPCENTER_SCORE_WIN_Z', '1'))
DEEPCENTER_SCORE_WIN_YX = int(os.environ.get('BIOHUB_DEEPCENTER_SCORE_WIN_YX', '2'))
DEEPCENTER_SCORE_CACHE_MAX_FRAMES = int(os.environ.get('BIOHUB_DEEPCENTER_SCORE_CACHE_MAX_FRAMES', '8'))
CONFIG_DISPLAY = {'experiment_tag': EXPERIMENT_TAG, 'method': METHOD, 'weights': WEIGHTS_RELATIVE, 'target_artifact_slug': TARGET_ARTIFACT_SLUG, 'primary_artifact_manifest': str(PRIMARY_ARTIFACT_MANIFEST), 'allow_artifact_fallback': ALLOW_ARTIFACT_FALLBACK, 'det_threshold': DET_THRESHOLD, 'unet_batch_size': UNET_BATCH_SIZE, 'use_ilp': USE_ILP, 'ilp_edge_weight': ILP_EDGE_WEIGHT, 'ilp_appearance_weight': ILP_APPEARANCE_WEIGHT, 'ilp_disappearance_weight': ILP_DISAPPEARANCE_WEIGHT, 'ilp_division_weight': ILP_DIVISION_WEIGHT, 'slice': SLICE, 'allow_pip_install': ALLOW_PIP_INSTALL, 'output_edge_max_um': OUTPUT_EDGE_MAX_UM, 'output_enforce_next_frame': OUTPUT_ENFORCE_NEXT_FRAME, 'output_single_parent_repair': OUTPUT_SINGLE_PARENT_REPAIR, 'output_single_child_repair': OUTPUT_SINGLE_CHILD_REPAIR, 'output_prune_isolated': OUTPUT_PRUNE_ISOLATED, 'output_motion_relink': OUTPUT_MOTION_RELINK, 'motion_relink_tight_um': MOTION_RELINK_TIGHT_UM, 'motion_relink_relaxed_um': MOTION_RELINK_RELAXED_UM, 'motion_relink_velocity_weight': MOTION_RELINK_VELOCITY_WEIGHT, 'motion_relink_learned_bonus': MOTION_RELINK_LEARNED_BONUS, 'motion_relink_max_frame_nodes': MOTION_RELINK_MAX_FRAME_NODES, 'output_division_geometry_filter': OUTPUT_DIVISION_GEOMETRY_FILTER, 'div_parent_max_um': DIV_PARENT_MAX_UM, 'div_sister_max_um': DIV_SISTER_MAX_UM, 'div_drop_to_single_if_bad': DIV_DROP_TO_SINGLE_IF_BAD, 'output_gap_close': OUTPUT_GAP_CLOSE, 'gap_close_max_gap': GAP_CLOSE_MAX_GAP, 'gap_close_effective_max_gap': min(GAP_CLOSE_MAX_GAP, 1), 'gap_close_um': GAP_CLOSE_UM, 'gap_density_adaptive': GAP_DENSITY_ADAPTIVE, 'gap_density_reference_um': GAP_DENSITY_REFERENCE_UM, 'gap_density_gain': GAP_DENSITY_GAIN, 'gap_density_max_step_delta_um': GAP_DENSITY_MAX_STEP_DELTA_UM, 'gap_density_neighbors': GAP_DENSITY_NEIGHBORS, 'gap_close_reuse_existing': GAP_CLOSE_REUSE_EXISTING, 'gap_close_reuse_um': GAP_CLOSE_REUSE_UM, 'gap_close_max_added_frac': GAP_CLOSE_MAX_ADDED_FRAC, 'gap_close_max_added_abs': GAP_CLOSE_MAX_ADDED_ABS, 'gap_refine_synthetic': GAP_REFINE_SYNTHETIC, 'gap_refine_win_z': GAP_REFINE_WIN_Z, 'gap_refine_win_yx': GAP_REFINE_WIN_YX, 'gap_refine_max_shift_um': GAP_REFINE_MAX_SHIFT_UM, 'output_filter_short_tracks': OUTPUT_FILTER_SHORT_TRACKS, 'output_min_track_len': OUTPUT_MIN_TRACK_LEN, 'output_keep_division_components': OUTPUT_KEEP_DIVISION_COMPONENTS, 'adaptive_short_track_rescue': ADAPTIVE_SHORT_TRACK_RESCUE, 'short_track_rescue_trigger_removed_frac': SHORT_TRACK_RESCUE_TRIGGER_REMOVED_FRAC, 'short_track_rescue_min_len': SHORT_TRACK_RESCUE_MIN_LEN, 'short_track_rescue_min_mean_edge_prob': SHORT_TRACK_RESCUE_MIN_MEAN_EDGE_PROB, 'short_track_rescue_max_mean_edge_dist_um': SHORT_TRACK_RESCUE_MAX_MEAN_EDGE_DIST_UM, 'short_track_rescue_max_nodes_frac': SHORT_TRACK_RESCUE_MAX_NODES_FRAC, 'short_track_rescue_max_nodes_abs': SHORT_TRACK_RESCUE_MAX_NODES_ABS, 'output_linefit_smooth': OUTPUT_LINEFIT_SMOOTH, 'output_linefit_weight': OUTPUT_LINEFIT_WEIGHT, 'output_linefit_window': OUTPUT_LINEFIT_WINDOW, 'output_gap2_recovery': OUTPUT_GAP2_RECOVERY, 'gap2_max_total_um': GAP2_MAX_TOTAL_UM, 'gap2_max_step_um': GAP2_MAX_STEP_UM, 'gap2_max_links_frac': GAP2_MAX_LINKS_FRAC, 'gap2_max_links_abs': GAP2_MAX_LINKS_ABS, 'gap2_require_context': GAP2_REQUIRE_CONTEXT, 'gap2_frame_frac_cap': GAP2_FRAME_FRAC_CAP, 'output_safe_divisions': OUTPUT_SAFE_DIVISIONS, 'safe_div_max_um': SAFE_DIV_MAX_UM, 'safe_div_sister_max_um': SAFE_DIV_SISTER_MAX_UM, 'safe_div_existing_child_max_um': SAFE_DIV_EXISTING_CHILD_MAX_UM, 'safe_div_frame_frac_cap': SAFE_DIV_FRAME_FRAC_CAP, 'safe_div_global_frac_cap': SAFE_DIV_GLOBAL_FRAC_CAP, 'use_deepcenter_add_only_gate': USE_DEEPCENTER_VETO, 'deepcenter_gap_add_gate': DEEPCENTER_GAP_VETO, 'deepcenter_safe_div_add_gate': DEEPCENTER_SAFE_DIV_VETO, 'deepcenter_gap_threshold': DEEPCENTER_GAP_THRESHOLD, 'deepcenter_expected_epoch': DEEPCENTER_EXPECTED_EPOCH, 'deepcenter_gap_confirm_min_span_um': DEEPCENTER_GAP_CONFIRM_MIN_SPAN_UM, 'deepcenter_safe_div_threshold': DEEPCENTER_SAFE_DIV_THRESHOLD, 'deepcenter_checkpoint_default': DEEPCENTER_CHECKPOINT_DEFAULT}
print('Biohub learned UNet + node-transformer + ILP submission')
print('COMP_DIR:', COMP_DIR, 'exists:', COMP_DIR.exists())
print('TEST_DIR:', TEST_DIR, 'exists:', TEST_DIR.exists())
print(json.dumps(CONFIG_DISPLAY, indent = 2, sort_keys = True))

import re

os.environ.setdefault('POLARS_PREFER_PKG', '32')

# Prepare required runtime dependencies and offline installation metadata
PACKAGE_SPECS = {'tracksdata': ('tracksdata', 'tracksdata'), 'zarr': ('zarr', 'zarr>=3.0.10,<4'), 'pyscipopt': ('pyscipopt', 'pyscipopt'), 'geff': ('geff', 'geff>=1.1.3.1.1'), 'geff_spec': ('geff_spec', 'geff-spec<1.2'), 'ilpy': ('ilpy', 'ilpy>=0.5.1'), 'polars': ('polars', 'polars>=1.36'), 'blosc2': ('blosc2', 'blosc2'), 'dask': ('dask', 'dask'), 'imagecodecs': ('imagecodecs', 'imagecodecs'), 'skimage': ('skimage', 'scikit-image>=0.24'), 'pyarrow': ('pyarrow', 'pyarrow'), 'rustworkx': ('rustworkx', 'rustworkx>=0.17.1'), 'sqlalchemy': ('sqlalchemy', 'sqlalchemy>=2'), 'numcodecs': ('numcodecs', 'numcodecs>=0.13,<0.16'), 'donfig': ('donfig', 'donfig>=0.8'), 'google_crc32c': ('google_crc32c', 'google-crc32c>=1.5'), 'bidict': ('bidict', 'bidict>=0.23.1'), 'psygnal': ('psygnal', 'psygnal>=0.14'), 'rich': ('rich', 'rich'), 'networkx': ('networkx', 'networkx>=3.2.1'), 'pydantic': ('pydantic', 'pydantic>=2.11'), 'pydantic_core': ('pydantic_core', 'pydantic-core'), 'annotated_types': ('annotated_types', 'annotated-types'), 'typing_extensions': ('typing_extensions', 'typing-extensions>=4.13'), 'typing_inspection': ('typing_inspection', 'typing-inspection'), 'markdown_it': ('markdown_it', 'markdown-it-py'), 'pygments': ('pygments', 'pygments'), 'click': ('click', 'click'), 'cloudpickle': ('cloudpickle', 'cloudpickle'), 'fsspec': ('fsspec', 'fsspec'), 'partd': ('partd', 'partd'), 'locket': ('locket', 'locket'), 'toolz': ('toolz', 'toolz'), 'yaml': ('yaml', 'pyyaml'), 'ndindex': ('ndindex', 'ndindex'), 'msgpack': ('msgpack', 'msgpack'), 'numexpr': ('numexpr', 'numexpr'), 'deprecated': ('deprecated', 'deprecated'), 'wrapt': ('wrapt', 'wrapt'), 'imageio': ('imageio', 'imageio'), 'PIL': ('PIL', 'pillow'), 'tifffile': ('tifffile', 'tifffile'), 'lazy_loader': ('lazy_loader', 'lazy-loader'), 'tqdm': ('tqdm', 'tqdm')}
EXTRA_SPECS_BY_NAME = {'tracksdata': ['bidict>=0.23.1', 'psygnal>=0.14', 'rich'], 'zarr': ['donfig>=0.8', 'google-crc32c>=1.5', 'numcodecs>=0.13,<0.16'], 'geff': ['geff-spec<1.2', 'networkx>=3.2.1', 'pydantic>=2.11', 'numcodecs>=0.13,<0.16'], 'geff_spec': ['pydantic>=2.11', 'annotated-types', 'pydantic-core', 'typing-inspection'], 'polars': ['polars-runtime-32'], 'dask': ['click', 'cloudpickle', 'fsspec', 'partd', 'pyyaml', 'toolz'], 'partd': ['locket'], 'blosc2': ['ndindex', 'msgpack', 'numexpr'], 'numcodecs': ['deprecated', 'msgpack', 'wrapt'], 'rich': ['markdown-it-py', 'pygments'], 'pydantic': ['annotated-types', 'pydantic-core', 'typing-extensions>=4.13', 'typing-inspection'], 'skimage': ['imageio', 'pillow', 'tifffile', 'lazy-loader', 'networkx']}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ['biohub-tracking-support-pack-v1']
ALLOW_PIP_INSTALL = os.environ.get('BIOHUB_ALLOW_PIP_INSTALL', '0') != '0'

# Check whether a required Python module is unavailable
def module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None

# Verify that a candidate directory contains the required repository and model weights
def has_model_artifact(path: Path) -> bool:
    has_repo_dir = (path / 'repo').exists()
    has_weights_dir = (path / 'weights' / METHOD / 'split_0' / 'edge_predictor_best.pth').exists()
    has_repo_zip = (path / 'repo.zip').exists()
    has_weights_zip = (path / 'weights.zip').exists()
    return has_repo_dir and has_weights_dir or (has_repo_zip and has_weights_zip)

# Load the artifact manifest when it is available
def artifact_manifest(path: Path) -> dict:
    manifest = path / 'ARTIFACT_MANIFEST.json'

    if not manifest.exists():
        return {}

    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}

# Check whether an artifact matches the requested model package
def artifact_matches_target(path: Path) -> bool:
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get('artifact_name', ''))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text

# Build the candidate artifact locations for a dataset slug
def candidate_roots_for_slug(slug: str) -> list[Path]:
    return [Path(f'/kaggle/input/datasets/pilkwang/{slug}'), Path(f'/kaggle/input/{slug}'), Path(f'/kaggle/input/{slug}/{slug}'), Path(f'PublicNotebook/{slug}')]

# Locate the primary BioHub model artifact across supported Kaggle paths
def find_artifacts_root() -> Path:
    candidates: list[Path] = []

    for env_name in ['BIOHUB_MODEL_ARTIFACTS', 'BIOHUB_ARTIFACTS']:
        explicit = os.environ.get(env_name, '').strip()

        if explicit:
            candidates.append(Path(explicit))
    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))

    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS:
            candidates.extend(candidate_roots_for_slug(slug))
    input_root = Path('/kaggle/input')

    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)

            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)

                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)
    seen: set[Path] = set()

    for candidate in candidates:
        candidate = candidate.expanduser()

        if candidate in seen:
            continue
        seen.add(candidate)

        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = '\n'.join((str(path) for path in candidates[:80]))
    raise FileNotFoundError(f'Could not find the required model artifact. Expected slug: {TARGET_ARTIFACT_SLUG}\nAttach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\nTo debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK = 1.\nChecked:\n' + checked)

# Check whether a directory contains installable offline package files
def _has_package_file(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    patterns = ('*.whl', '*.tar.gz', '*.zip')
    return any((any(path.glob(pattern)) for pattern in patterns))

# Collect directories that contain offline dependency packages
def find_offline_package_dirs(artifacts: Path) -> list[Path]:
    candidates: list[Path] = [artifacts / 'wheels', artifacts, Path('/kaggle/working'), Path('/kaggle/working/wheels')]
    input_root = Path('/kaggle/input')

    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / 'wheels', child])

                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / 'wheels', grandchild])
    out: list[Path] = []
    seen: set[Path] = set()

    for candidate in candidates:
        candidate = candidate.expanduser()

        if candidate in seen:
            continue
        seen.add(candidate)

        if _has_package_file(candidate):
            out.append(candidate)
    return out

# Remove selected imported modules so refreshed packages can be reloaded cleanly
def purge_imported_modules(package_names: list[str]) -> None:
    roots = {'tracksdata'}

    for name in package_names:
        if name in PACKAGE_SPECS:
            module = PACKAGE_SPECS[name][0]
            roots.add(module.split('.')[0])

        if name == 'polars':
            roots.add('polars')

    for root in roots:
        for module_name in list(sys.modules):
            if module_name == root or module_name.startswith(root + '.'):
                sys.modules.pop(module_name, None)

# Verify that the installed Polars runtime is compatible with the pipeline
def polars_runtime_ready() -> bool:
    try:
        import polars as _pl
        from polars._plr import PySeries as _PySeries
        _ = _PySeries
        return hasattr(_pl, 'Float16') and _pl.Series([-999999.0], dtype = _pl.Float64).dtype == _pl.Float64
    except Exception:
        return False

# Identify installed packages that need a compatible runtime refresh
def packages_requiring_refresh() -> list[str]:
    refresh: list[str] = []

    if not module_missing('polars') and (not polars_runtime_ready()):
        refresh.append('polars')

    if not module_missing('zarr'):
        try:
            import zarr as _zarr
            version_text = str(getattr(_zarr, '__version__', '0'))
            major = int(version_text.split('.', 1)[0])

            if major < 3:
                refresh.append('zarr')
        except Exception:
            refresh.append('zarr')
    return refresh

# Build the dependency specification list for missing or incompatible packages
def dependency_specs_for(missing: list[str]) -> list[str]:
    specs: list[str] = []
    seen: set[str] = set()

    # Add a dependency specification once while preserving order
    def add(spec: str) -> None:
        key = spec.lower()

        if key not in seen:
            seen.add(key)
            specs.append(spec)

    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])

        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs

# Collect import errors for required runtime modules
def import_failures() -> dict[str, str]:
    failures: dict[str, str] = {}

    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f'{type(exc).__name__}: {exc}'
    return failures

# Map import failures back to package names that need installation
def missing_names_from_failures(failures: dict[str, str]) -> list[str]:
    names: list[str] = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}

    for message in failures.values():
        match = re.search('No module named [\'\\"]([^\'\\"]+)[\'\\"]', message)

        if match:
            module = match.group(1).split('.')[0]
        else:
            match = re.search('module [\'\\"]([^\'\\"]+)[\'\\"] has no attribute', message)

            if not match:
                continue
            module = match.group(1).split('.')[0]
        name = module_to_name.get(module)

        if name and name not in names:
            names.append(name)
    return names

# Install missing dependencies from offline wheels or the allowed fallback source
def install_missing_dependencies(missing: list[str], artifacts: Path) -> None:
    specs = dependency_specs_for(missing)
    force_reinstall = bool({'polars', 'zarr'} & set(missing))

    if not specs:
        return
    package_dirs = find_offline_package_dirs(artifacts)

    if package_dirs:
        offline_cmd = [sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps']

        if force_reinstall:
            offline_cmd.append('--force-reinstall')

        for package_dir in package_dirs:
            offline_cmd.extend(['--find-links', str(package_dir)])
        offline_cmd.extend(specs)
        print('Installing missing packages from offline package dirs:', missing)
        print('Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.')
        print('Offline package dirs:', [str(path) for path in package_dirs])
        result = subprocess.run(offline_cmd, text = True, capture_output = True)

        if result.returncode == 0:
            purge_imported_modules(missing)
            print('Offline dependency install succeeded.')
            return
        print('Offline dependency install failed. Last pip output:')
        print((result.stdout or '')[-2000:])
        print((result.stderr or '')[-2000:])

    if ALLOW_PIP_INSTALL:
        online_cmd = [sys.executable, '-m', 'pip', 'install', '--no-deps']

        if force_reinstall:
            online_cmd.append('--force-reinstall')
        online_cmd.extend(specs)
        print('Installing missing packages from PyPI:', missing)
        result = subprocess.run(online_cmd, text = True, capture_output = True)

        if result.returncode == 0:
            purge_imported_modules(missing)
            print('PyPI dependency install succeeded.')
            return
        print('PyPI dependency install failed. Last pip output:')
        print((result.stdout or '')[-2000:])
        print((result.stderr or '')[-2000:])
    command = 'pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs'
    raise ImportError('Missing required packages or dependency wheels: ' + ', '.join(missing) + '\nAttach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\n' + command + '\nDo not quote zarr>=3.0.10,<4 in Kaggle dependency input.')

# Resolve and verify all runtime dependencies before inference starts
def ensure_dependencies(artifacts: Path) -> None:
    for _ in range(5):
        refresh = packages_requiring_refresh()

        if refresh:
            install_missing_dependencies(refresh, artifacts)
            continue
        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]

        if missing:
            install_missing_dependencies(missing, artifacts)
            continue
        failures = import_failures()

        if not failures:
            print('Required graph/Zarr/ILP packages import successfully.')
            return
        missing_from_import = missing_names_from_failures(failures)

        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts)
            continue
        raise ImportError('Required packages are present but failed to import. This may indicate a binary dependency mismatch in the live notebook kernel. Keep Kaggle dependency input empty and attach the wheels artifact.\n' + json.dumps(failures, indent = 2))
    failures = import_failures()
    raise ImportError('Dependency recovery did not converge after repeated offline installs. The attached support artifact may be missing wheels.\n' + json.dumps(failures, indent = 2))

# Remove an existing file, symlink, or directory before materialization
def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)

# Materialize a directory tree by copying a folder or extracting its archive
def copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:
    remove_path(dst)

    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst)
        return

    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents = True, exist_ok = True)

        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f'Missing source tree or zip: {src_dir} / {src_zip}')

# Reuse an existing directory through a symlink with a copy fallback
def link_or_copy_tree(src: Path, dst: Path) -> None:
    remove_path(dst)

    try:
        os.symlink(src, dst, target_is_directory = True)
    except Exception:
        shutil.copytree(src, dst)

# Prepare the inference repository and model weights inside the working directory
def materialize_inference_repo(artifacts: Path) -> None:
    copy_or_extract_tree(artifacts / 'repo', artifacts / 'repo.zip', REPO_DIR)
    weights_src = artifacts / 'weights'
    weights_zip = artifacts / 'weights.zip'
    weights_dst = REPO_DIR / 'weights'

    if weights_src.exists() and weights_src.is_dir():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists() and weights_zip.is_file():
        remove_path(weights_dst)
        weights_dst.mkdir(parents = True, exist_ok = True)

        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f'Missing weights tree or zip under {artifacts}')
    required = [REPO_DIR / 'scripts' / 'predict_unet_transformer.py', REPO_DIR / WEIGHTS_RELATIVE]
    missing = [str(path) for path in required if not path.exists()]

    if missing:
        raise FileNotFoundError('Materialized inference repo is incomplete:\n' + '\n'.join(missing))
    print('Inference repo:', REPO_DIR)
    print('Weights:', REPO_DIR / WEIGHTS_RELATIVE)

os.environ['BIOHUB_DEEPCENTER_CHECKPOINT'] = '/kaggle/input/datasets/reyhanksatria/biohub-deepcenterunet3d-center-prior-v1/weights/full_frame_center/best.pt'

# Locate the primary artifact and materialize all required runtime dependencies
ARTIFACTS = find_artifacts_root()
print('ARTIFACTS:', ARTIFACTS)
print('Has offline wheels:', (ARTIFACTS / 'wheels').exists())
manifest_info = artifact_manifest(ARTIFACTS)

if manifest_info:
    print('Artifact name:', manifest_info.get('artifact_name'))
    print('Weight sha256:', manifest_info.get('model', {}).get('weight_sha256'))
    print('Weight path:', manifest_info.get('model', {}).get('weight_path'))
    _expected_primary_sha256 = '12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771'
    _actual_primary_sha256 = str(manifest_info.get('model', {}).get('weight_sha256', ''))

    if _actual_primary_sha256 != _expected_primary_sha256:
        raise RuntimeError(f"Primary model checksum mismatch: expected {_expected_primary_sha256}, got {_actual_primary_sha256 or 'missing'}")

ensure_dependencies(ARTIFACTS)
materialize_inference_repo(ARTIFACTS)

import hashlib as _integrity_hashlib

# Verify the support repository and model checkpoints before dynamic source patching
_support_expected_sha256 = {'scripts/augmentations.py': '13db09817bf492f8d0f710a0a4d09776320b262060167055090a303fc6057f4e', 'scripts/dataspec.py': 'e69bf952fb985477ac50ff8598a35020c95d20a035a09b81ab4056e655dd311f', 'scripts/evaluate.py': '614813cc51c3581c6ccda4bb20725a19da8ecac4a27620654bfca58319cffa3c', 'scripts/predict_unet_transformer.py': 'c44e771ba5980b820f93091e03a303c25dfe8f3232e501f54dc9565731c234b9', 'scripts/train_unet_transformer.py': 'c4f6317736bb3bb1ec8f3f6e9a6d935a463e3f0f1f685481b2d13218d35dc9ea', 'src/biohub_tracking/__init__.py': '26a18d8da84e40da73281a48ebc3017d847a2e57431ab63e8629d2109e6e8571', 'src/biohub_tracking/division_metrics.py': 'd1cf1e0a43009d02174f1699ce2aa28458a2220ac4b521731d3bcf31cf8c76be', 'src/biohub_tracking/img_proc.py': '00e8ef0adc8b39f1aaaa547ea6197b906bf9e8c009e339d3e95f8f8dbf31be3f', 'src/biohub_tracking/io.py': 'efae135b088cecaab463d889f16c885ef6da3ad27b0747327d8ddc28d866b7bd', 'src/biohub_tracking/metrics.py': '31baf45b54c78f68bab4f65dd8f4b38bca702abb644171c6df7c46cdeef55d83', 'src/biohub_tracking/models/__init__.py': 'ab7587ef79856bae50d24b62e5805092d0459ee1c586522b763f9ef70c093e1d', 'src/biohub_tracking/models/simple_node_transformer.py': 'b97209edeb03840e80d903e3e2a8c81c520641c8ef343f6ca2904d0f80db064e', 'src/biohub_tracking/models/temporal_unet.py': 'd809c35d42f504161074ddeaaa7aee5b407e5bca7f9b4e1d5f9b2ff345666cac'}
_support_expected_manifest_sha256 = '978b626d1fd1e7397435a437dfe68691defe1572fc3c20e61012d7c9b52ed029'
_primary_expected_sha256 = '12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771'
_deepcenter_expected_sha256 = '8040999a92f6b7bbd98fa8cf458141e045c0f9ad7c936bdb3b18e1f7edafe2a0'

# Compute a SHA256 checksum for runtime integrity validation
def _integrity_sha256_file(path: Path) -> str:
    digest = _integrity_hashlib.sha256()

    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

_support_materialized_paths = {path.relative_to(REPO_DIR).as_posix(): path for path in REPO_DIR.rglob('*.py')}
_support_actual_names = set(_support_materialized_paths)
_support_expected_names = set(_support_expected_sha256)

if _support_actual_names != _support_expected_names:
    raise RuntimeError({'support_repo_python_files_missing': sorted(_support_expected_names - _support_actual_names), 'support_repo_python_files_extra': sorted(_support_actual_names - _support_expected_names)})

_support_actual_sha256 = {relative: _integrity_sha256_file(_support_materialized_paths[relative]) for relative in sorted(_support_materialized_paths)}

if _support_actual_sha256 != _support_expected_sha256:
    raise RuntimeError({'support_repo_python_checksum_mismatch': {relative: {'expected': _support_expected_sha256[relative], 'actual': _support_actual_sha256[relative]} for relative in sorted(_support_expected_sha256) if _support_actual_sha256[relative] != _support_expected_sha256[relative]}})

_support_manifest_bytes = ''.join((f'{_support_actual_sha256[relative]}  {relative}\n' for relative in sorted(_support_actual_sha256))).encode('utf-8')
_support_actual_manifest_sha256 = _integrity_hashlib.sha256(_support_manifest_bytes).hexdigest()

if _support_actual_manifest_sha256 != _support_expected_manifest_sha256:
    raise RuntimeError(f'Support repo manifest checksum mismatch: expected {_support_expected_manifest_sha256}, got {_support_actual_manifest_sha256}')

_primary_materialized_path = REPO_DIR / WEIGHTS_RELATIVE
_primary_actual_sha256 = _integrity_sha256_file(_primary_materialized_path)

if _primary_actual_sha256 != _primary_expected_sha256:
    raise RuntimeError(f'Materialized primary model checksum mismatch: expected {_primary_expected_sha256}, got {_primary_actual_sha256}')


# The elastic augmentation, verbatim from notebooks/elastic_augment.py
ELASTIC_SOURCE = '"""Smooth random in-plane deformation for `train_unet_transformer.py`.\n\nThe support pack trains with exactly two augmentations::\n\n    DEFAULT_AUGMENTATIONS = [brightness_augment, flip_augment]\n\n`flip_augment` samples the eight axis-aligned symmetries and `brightness_augment` adds a\nscalar. That is the whole of it -- no deformation, no affine, no noise. hengck23 suggested\nelastic deformation in July and nobody in this lineage has tried it, because nobody in this\nlineage trains.\n\n**The failure mode this file is written around.** An augmentation here receives the images\n*and the node coordinates*. Warp the image and leave the coordinates behind and every label\nis silently wrong: training converges, the loss falls, the checkpoint is garbage, and nothing\nanywhere says so. That is worse than a crash, so `elastic_augment` ends by *measuring* whether\nthe coordinates followed the image and raising if they did not.\n\nDesign choices, each for a reason:\n\n* **In-plane only.** `downsample = (1, 4, 4)` -- Z is not downsampled and its voxels are\n  1.625 um against 0.40625 in Y and X. A deformation field over Z would mix physically\n  different scales; over Y and X it does not.\n* **One field for the whole window.** `window_size = 2`, and the association head learns\n  correspondence *between* the two frames. Warping them differently would teach it motion\n  that is not there. The same field is applied to every frame and every z-slice.\n* **First-order coordinate update.** `grid_sample` with grid ``p + d(p)`` produces\n  ``out(p) = in(p + d(p))``, so content at input ``q`` lands near output ``q - d(q)``. For a\n  field this smooth and this small the first-order update is accurate to well under a voxel,\n  and the intensity check at the end is what proves it.\n"""\n# this file is APPENDED to the pack\'s scripts/augmentations.py, so it must not open with\n# anything that has to come first in a file. `from __future__ import annotations` did, and\n# landed at line 98 of the concatenation: SyntaxError, run dead in ninety seconds. The same\n# rule bit the notebook builder an hour earlier and I did not carry the lesson across the\n# two places the same text is used. Nothing here needs it -- the annotations are all\n# builtin generics, valid at runtime since 3.9.\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\n\ndef elastic_augment(\n    imgs: torch.Tensor,\n    coords: torch.Tensor,\n    masks: torch.Tensor,\n    *,\n    rng: np.random.Generator,\n    max_shift_vox: float = 3.0,\n    control: int = 4,\n    prob: float = 0.5,\n    check_margin: float = 0.05,\n) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n    """Deform Y and X by a smooth random field, carrying the node coordinates with it.\n\n    Parameters\n    ----------\n    imgs : torch.Tensor\n        ``(W, Z, Y, X)`` normalised images.\n    coords : torch.Tensor\n        ``(W, M, 3)`` node coordinates in voxels, ordered ``(z, y, x)``. Padded rows are\n        zero and excluded by ``masks``; they are left untouched, exactly as `flip_augment`\n        leaves them.\n    masks : torch.Tensor\n        ``(W, M)`` boolean, true for real nodes.\n    rng : np.random.Generator\n        The trainer\'s generator, so a run stays reproducible from ``base_seed``.\n    max_shift_vox : float\n        Largest displacement anywhere in the field, in Y/X voxels. At the pack\'s\n        ``downsample = (1, 4, 4)`` one voxel is 1.625 um, so 3 voxels is about a cell radius.\n    control : int\n        Side of the coarse control grid the field is drawn on before upsampling. Small values\n        give long-wavelength warps; large ones approach noise, which would fight the detector\n        rather than regularise it.\n    prob : float\n        Probability of applying the deformation at all.\n    check_margin : float\n        How much better the updated coordinates must score than the un-updated ones on the\n        warped image. A skipped update scores exactly the same; a correct one is far ahead.\n    """\n    if rng.random() >= prob:\n        return imgs, coords, masks\n\n    W, Z, Y, X = imgs.shape\n    dev, dt = imgs.device, imgs.dtype\n\n    # A smooth field over (Y, X), shared by every frame and every slice.\n    coarse = torch.from_numpy(\n        rng.normal(0.0, 1.0, size=(1, 2, control, control)).astype("float32")\n    )\n    field = F.interpolate(coarse, size=(Y, X), mode="bicubic", align_corners=True)\n    peak = field.abs().amax().clamp_min(1e-6)\n    field = (field / peak) * float(max_shift_vox)          # (1, 2, Y, X) in voxels, [dy, dx]\n    field = field.to(device=dev, dtype=torch.float32)\n\n    # Sampling grid: output pixel p reads input p + d(p). grid_sample wants (x, y) last,\n    # normalised to [-1, 1] with align_corners=True.\n    yy, xx = torch.meshgrid(\n        torch.arange(Y, device=dev, dtype=torch.float32),\n        torch.arange(X, device=dev, dtype=torch.float32),\n        indexing="ij",\n    )\n    src_y = yy + field[0, 0]\n    src_x = xx + field[0, 1]\n    norm = lambda v, n: (2.0 * v / max(n - 1, 1)) - 1.0\n    grid = torch.stack([norm(src_x, X), norm(src_y, Y)], dim=-1)[None]    # (1, Y, X, 2)\n\n    flat = imgs.reshape(W * Z, 1, Y, X).to(torch.float32)\n    warped = F.grid_sample(\n        flat, grid.expand(W * Z, -1, -1, -1),\n        mode="bilinear", padding_mode="border", align_corners=True,\n    )\n    out_imgs = warped.reshape(W, Z, Y, X).to(dt)\n\n    # Content at input q lands near output q - d(q), so sample the field AT the node and\n    # subtract. Nearest-voxel lookup is enough: the field varies over hundreds of voxels.\n    out_coords = coords.clone()\n    if masks.any():\n        cy = coords[..., 1].round().long().clamp_(0, Y - 1)\n        cx = coords[..., 2].round().long().clamp_(0, X - 1)\n        dy = field[0, 0][cy, cx]\n        dx = field[0, 1][cy, cx]\n        m = masks.to(torch.bool)\n        out_coords[..., 1] = torch.where(m, coords[..., 1] - dy.to(coords.dtype), coords[..., 1])\n        out_coords[..., 2] = torch.where(m, coords[..., 2] - dx.to(coords.dtype), coords[..., 2])\n        out_coords[..., 1].clamp_(0, Y - 1)\n        out_coords[..., 2].clamp_(0, X - 1)\n\n        # Did the coordinates follow the image? Compare the warped image sampled at the\n        # UPDATED coordinates against the same warped image sampled at the ORIGINAL ones.\n        #\n        # The first version compared before-warp against after-warp and fired on real data at\n        # contrast 108.3 -> 63.1, a false positive: at downsample (1, 4, 4) a cell is barely\n        # a voxel across in Y and X, and bilinear resampling of a one-voxel peak loses ~40%\n        # of its amplitude no matter how right the coordinates are. My synthetic test used\n        # sigma=2 blobs and lost only 13%, which is exactly why it passed.\n        #\n        # Measuring both terms on the SAME warped image removes interpolation loss from the\n        # comparison entirely. A skipped update makes the two identical by construction; a\n        # wrong-signed one makes the updated coordinates worse than the originals. Only a\n        # correct update is clearly better, and only when the field actually moved something,\n        # which is why the check is gated on a real displacement.\n        # Gated on the displacement the field INTENDED at the nodes, not on the one the\n        # coordinates actually moved. Gating on the realised shift is circular: an update\n        # that never happened leaves the shift at zero, the gate never opens, and the check\n        # passes -- which is what the first rewrite did, silently, in six of six test cases.\n        # Gated on whether the update actually moves nodes into DIFFERENT VOXELS. `_contrast`\n        # samples at the nearest voxel, so a sub-voxel warp leaves `good` and `stale` reading\n        # the same voxels and the comparison has nothing to measure: a real 0.78-voxel warp\n        # scored 2.234 against 2.218, a 0.7% difference against a 5% margin, and the guard\n        # failed the run. Below about a voxel this instrument is blind, and a blind check\n        # must abstain rather than accuse.\n        # Per node, not max against max. `out_coords` is clamped to the volume, so a node on\n        # the border does not move the full distance the field asks for -- and if that node\n        # happens to carry the largest displacement, `max(realised)` falls well below\n        # `max(intended)` with nothing wrong. That false-fired at "2.37 voxels asked, 1.02\n        # moved", after two clean epochs. My test put every node 8-10 voxels from the edge,\n        # so it could not see it: the test was easier than the data, for the second time.\n        want = torch.stack([dy.abs(), dx.abs()], -1)\n        got = (out_coords[..., 1:] - coords[..., 1:]).abs()\n        agree = (torch.isclose(got, want, atol=1e-3) | (want < 1e-6))[m].all(-1)\n        frac = float(agree.to(torch.float32).mean())\n        intended = float(torch.maximum(dy[m].abs().amax(), dx[m].abs().amax()))\n        realised = float((out_coords - coords).abs().amax())\n        moved = float((out_coords.round() != coords.round()).any(-1)[m].to(torch.float32).mean())\n        # TWO checks, deliberately gated on different things. Nesting them under one gate is\n        # how this guard has been wrong three times: gate on any property of the OUTPUT and a\n        # skipped update closes the gate on itself.\n        #\n        # A -- did the update happen at all? Gated only on the field, which the update cannot\n        # influence. Catches a skipped update at every warp magnitude.\n        if intended > 0.5 and frac < 0.9:\n            raise RuntimeError(\n                f"elastic_augment: only {frac:.0%} of nodes moved by the displacement the "\n                f"field asked for (up to {intended:.2f} voxels, largest realised "\n                f"{realised:.2f}) -- the update did not happen."\n            )\n\n        # B -- did it move them the right way? Needs the nodes in different voxels before\n        # `_contrast` can see anything, so below about a voxel this abstains. A wrong sign\n        # under a sub-voxel warp goes undetected, which is the honest limit of the check and\n        # is bounded by A having already proved the update ran.\n        if intended > 0.5 and moved > 0.25:\n            good = _contrast(out_imgs, out_coords, m)\n            stale = _contrast(out_imgs, coords, m)\n            if good < stale * (1.0 + check_margin):\n                # WARNS, it does not raise -- and that is the fifth version of this check.\n                # Every batch-level attempt to police DIRECTION has false-fired, because the\n                # margin available depends on how far the warp happens to move nodes on that\n                # batch: a real 0.60-voxel warp separates 4.265 from 4.174, 2%, against any\n                # threshold big enough to mean something. Policing direction per batch is\n                # measuring a thing this instrument cannot resolve.\n                #\n                # Direction is instead settled ONCE, by `self_test()` below, on a phantom\n                # with a warp large enough to be unambiguous. What stays fatal here is check\n                # A, which is exact and cannot false-fire. A heuristic that is blind by\n                # construction must not be able to stop a twelve-hour run.\n                print(\n                    f"elastic_augment: weak direction signal -- {realised:.2f} voxel warp, "\n                    f"updated {good:.3f} vs original {stale:.3f}",\n                    flush=True,\n                )\n    return out_imgs, out_coords, masks\n\n\ndef _contrast(imgs: torch.Tensor, coords: torch.Tensor, mask: torch.Tensor) -> float:\n    """Mean intensity at the masked node coordinates, divided by mean image intensity.\n\n    Nodes are cell centres, so this is comfortably above 1 whenever the coordinates point at\n    cells and falls to about 1 when they point anywhere else.\n    """\n    W, Z, Y, X = imgs.shape\n    w_idx = torch.arange(W, device=imgs.device)[:, None].expand_as(mask)[mask]\n    cz = coords[..., 0][mask].round().long().clamp_(0, Z - 1)\n    cy = coords[..., 1][mask].round().long().clamp_(0, Y - 1)\n    cx = coords[..., 2][mask].round().long().clamp_(0, X - 1)\n    at_nodes = float(imgs[w_idx, cz, cy, cx].to(torch.float32).mean())\n    overall = float(imgs.to(torch.float32).mean())\n    return at_nodes / overall if abs(overall) > 1e-9 else 1.0\n\n\ndef self_test(seed: int = 0) -> None:\n    """Prove the coordinate update tracks the image, once, where the answer is unambiguous.\n\n    Called at import time by the training notebook. A large warp on a synthetic phantom puts\n    the nodes several voxels from where they started, which is the regime the contrast\n    comparison can actually resolve -- unlike a random training batch, where the field may\n    move nothing far enough to measure. Raising here costs a second; raising per batch has\n    cost five runs.\n    """\n    rng = np.random.Generator(np.random.PCG64(seed))\n    W, Z, Y, X, M = 2, 8, 96, 96, 40\n    cz = rng.integers(1, Z - 1, M)\n    cy = rng.integers(10, Y - 10, M)\n    cx = rng.integers(10, X - 10, M)\n    zz, yy, xx = torch.meshgrid(\n        *[torch.arange(n, dtype=torch.float32) for n in (Z, Y, X)], indexing="ij")\n    base = sum(torch.exp(-(((zz - cz[i]) / 1.2) ** 2\n                           + ((yy - cy[i]) / 0.8) ** 2\n                           + ((xx - cx[i]) / 0.8) ** 2)) for i in range(M))\n    imgs = (base + 0.3)[None].repeat(W, 1, 1, 1)\n    coords = torch.tensor(np.stack([cz, cy, cx], -1), dtype=torch.float32)[None].repeat(W, 1, 1)\n    masks = torch.ones(W, M, dtype=torch.bool)\n\n    out_imgs, out_coords, _ = elastic_augment(\n        imgs, coords, masks, rng=np.random.Generator(np.random.PCG64(seed + 1)),\n        prob=1.0, max_shift_vox=6.0)\n    good = _contrast(out_imgs, out_coords, masks)\n    stale = _contrast(out_imgs, coords, masks)\n    shift = float((out_coords - coords).abs().amax())\n    if not (shift > 2.0 and good > stale * 1.25):\n        raise RuntimeError(\n            f"elastic_augment self-test FAILED: {shift:.2f} voxel warp, updated coordinates "\n            f"score {good:.3f} against {stale:.3f} for the originals"\n        )\n    print(f"elastic_augment self-test OK: {shift:.2f} voxel warp, updated {good:.3f} "\n          f"vs original {stale:.3f}", flush=True)\n'


# ==========================================================================
# CLAUDE FINE-TUNE -- everything above this line is the support pack's own
# setup, taken verbatim from the 0.946 notebook and left unmodified.
# ==========================================================================
import random as _rnd

TRAIN_DIR = COMP_DIR / 'train'
if not TRAIN_DIR.exists():
    raise FileNotFoundError(f'No training data mounted at {TRAIN_DIR}')
_stems = sorted(p.name[:-5] for p in TRAIN_DIR.iterdir() if p.name.endswith('.zarr'))
print(f'Training movies mounted: {len(_stems)}', flush=True)
if len(_stems) < 50:
    raise RuntimeError(f'expected ~199 training movies, found {len(_stems)}')

# A holdout the checkpoint we are fine-tuning has ALREADY SEEN. notes/72 section 3: the
# public checkpoint trained on all 199 movies, so this comparison is biased in ITS favour --
# which is what makes a win on it real rather than a training-set artifact. It is the first
# honest validation signal this project has had.
_HOLDOUT = int(os.environ.get('BIOHUB_TRAIN_HOLDOUT', '30'))
_shuffled = list(_stems)
_rnd.Random(20260912).shuffle(_shuffled)
_val, _tr = sorted(_shuffled[:_HOLDOUT]), sorted(_shuffled[_HOLDOUT:])
_splits = REPO_DIR / 'claude_finetune_splits.json'
_splits.write_text(json.dumps([{'split': 0, 'train': _tr, 'test': _val}], indent=2))
print(f'Fold 0: {len(_tr)} train / {len(_val)} holdout', flush=True)
print('holdout:', _val[:6], '...', flush=True)

# ---- the augmentation the pack does not have ----------------------------
_AUG = REPO_DIR / 'scripts' / 'augmentations.py'
_aug_src = _AUG.read_text()
if 'def elastic_augment' in _aug_src:
    raise RuntimeError('augmentations.py already defines elastic_augment')
_AUG.write_text(_aug_src + "\n\n" + ELASTIC_SOURCE)

_T = REPO_DIR / 'scripts' / 'train_unet_transformer.py'
_t = _T.read_text()
_edits = [
    ('from augmentations import brightness_augment, flip_augment\n',
     'from augmentations import brightness_augment, flip_augment, elastic_augment\n'),
    ('DEFAULT_AUGMENTATIONS = [brightness_augment, flip_augment]',
     'DEFAULT_AUGMENTATIONS = [brightness_augment, flip_augment, elastic_augment]'),
    # `WEIGHTS_PATH` comes from `dataspec` and lands inside the materialised repo, whose
    # `weights/` is a link into the read-only `/kaggle/input` mount -- the pack's own
    # checkpoints live there, so it never needed to be writable for inference. Training is
    # the first thing that writes to it: "OSError: [Errno 30] Read-only file system", after
    # the run had already loaded 169 movies. Redirect it somewhere we own.
    ('from dataspec import WEIGHTS_PATH\n',
     'from dataspec import WEIGHTS_PATH  # noqa: F401\n'
     'WEIGHTS_PATH = Path("/kaggle/working/claude_weights")\n'
     'WEIGHTS_PATH.mkdir(parents=True, exist_ok=True)\n'),
    # `--unet-weights` restores the BACKBONE ONLY, and from this checkpoint it restores
    # nothing at all. The trainer does
    #
    #     unet = TemporalUNet3D(...)
    #     unet.load_state_dict(torch.load(unet_weights), strict=False)
    #
    # while `edge_predictor_best.pth` was saved from the whole `UNetNodeTransformer`, so its
    # keys are `unet.enc...`, `node_transformer...`, `edge_head...`. Loaded into a bare
    # TemporalUNet3D every one of those is "unexpected" and every backbone parameter is
    # "missing": strict=False turns a total mismatch into a silent no-op, and the run becomes
    # a from-scratch training that looks exactly like a fine-tune. The flag is not wrong --
    # it is for a UNet-only pretrain -- it is the wrong flag for this file.
    #
    # Restore the FULL model instead, association head included, and refuse to continue if
    # the restore did not actually take.
    ("""    model = UNetNodeTransformer(
        unet=unet,
        unet_out_channels=unet_out_channels,
        pos_feat_dim=pos_feat_dim,
    ).to(device)
""",
     """    model = UNetNodeTransformer(
        unet=unet,
        unet_out_channels=unet_out_channels,
        pos_feat_dim=pos_feat_dim,
    ).to(device)

    if unet_weights is not None:
        _full = torch.load(unet_weights, map_location="cpu", weights_only=True)
        _missing, _unexpected = model.load_state_dict(_full, strict=False)
        _restored = len(_full) - len(_unexpected)
        print(f"  FULL restore from {unet_weights}: {_restored}/{len(_full)} tensors "
              f"loaded, {len(_missing)} left at init, {len(_unexpected)} unused",
              flush=True)

        if _restored < len(_full) // 2 and os.environ.get(
                "BIOHUB_TRAIN_ALLOW_SCRATCH", "0") == "0":
            raise RuntimeError(
                f"only {_restored} of {len(_full)} checkpoint tensors matched the model -- "
                "this would train from scratch while looking like a fine-tune"
            )
"""),
    # The most valuable line in the run: score the checkpoint we are about to fine-tune on
    # the holdout BEFORE touching it, so every epoch after is measured against it.
    ('    for epoch in pbar:\n        t0 = time.monotonic()\n',
     '    _b_loss, _b_acc, _b_recall = evaluate(model, test_loader, device,'
     ' pool_kernel_um=pool_kernel_um)\n'
     '    print(f"  BASELINE epoch -1 (public checkpoint, no training) | "\n'
     '          f"acc={_b_acc:.4f} | recall={_b_recall:.4f} | score={_b_acc * _b_recall:.4f}",\n'
     '          flush=True)\n'
     '    best_score = _b_acc * _b_recall\n'
     '    for epoch in pbar:\n        t0 = time.monotonic()\n'),
]
for _old, _new in _edits:
    if _t.count(_old) != 1:
        raise RuntimeError(f'trainer patch matched {_t.count(_old)}x, expected 1: {_old[:60]!r}')
    _t = _t.replace(_old, _new, 1)
compile(_t, str(_T), 'exec')
_T.write_text(_t)

# ---- make the temporal attention launchable on an sm_60 card -------------
# `_TemporalAttention.forward` flattens to (B * S, T, C) with S the whole downsampled
# volume, so attention runs over millions of length-T sequences in ONE call. cuBLAS batched
# GEMM takes its batch count as a CUDA grid dimension capped at 65535, and past that a P100
# answers `invalid configuration argument` -- which is what killed the run after it had
# already loaded all 169 movies. Slicing that batch is mathematically identical: every
# sequence attends only across its own T timesteps, so there is nothing between slices to
# lose. T is 2, so the loop costs seconds an epoch and lowers peak memory as well.
_TU = REPO_DIR / 'src' / 'biohub_tracking' / 'models' / 'temporal_unet.py'
_tu = _TU.read_text()
_attn_old = ("        h = x.reshape(B, T, C, S).permute(0, 3, 1, 2).reshape(B * S, T, C)\n"
             "        h = self.norm(h)\n"
             "        h, _ = self.attn(h, h, h, need_weights=False)\n")
_attn_new = ("        h = x.reshape(B, T, C, S).permute(0, 3, 1, 2).reshape(B * S, T, C)\n"
             "        h = self.norm(h)\n"
             "        _chunk = int(os.environ.get('BIOHUB_ATTN_CHUNK', '32768'))\n"
             "\n"
             "        if _chunk <= 0 or h.shape[0] <= _chunk:\n"
             "            h, _ = self.attn(h, h, h, need_weights=False)\n"
             "        else:\n"
             "            _parts = []\n"
             "\n"
             "            for _i in range(0, h.shape[0], _chunk):\n"
             "                _p = h[_i:_i + _chunk]\n"
             "                _parts.append(self.attn(_p, _p, _p, need_weights=False)[0])\n"
             "            h = torch.cat(_parts, dim=0)\n"
             "            del _parts\n")
if _tu.count(_attn_old) != 1:
    raise RuntimeError(f'attention patch matched {_tu.count(_attn_old)}x, expected 1')
_tu = _tu.replace(_attn_old, _attn_new, 1).replace('import math\n', 'import math\nimport os\n', 1)
compile(_tu, str(_TU), 'exec')
_TU.write_text(_tu)
print('temporal attention chunked at', os.environ.get('BIOHUB_ATTN_CHUNK', '32768'),
      'sequences per launch', flush=True)
print('elastic_augment installed; baseline eval added; best_score seeded from the baseline '
      'so nothing worse than the public checkpoint can be saved', flush=True)

# Settle the coordinate-update direction ONCE, in the interpreter the trainer will use, on a
# phantom warped far enough for the answer to be unambiguous. Five runs died to a per-batch
# version of this check, because the margin it needs depends on how far a random field
# happens to move nodes on that batch. Here the warp is chosen, so it does not.
_st = subprocess.run(
    ['/usr/bin/python3', '-c',
     'import sys; sys.path.insert(0, "scripts"); from augmentations import self_test; '
     'self_test()'],
    cwd=REPO_DIR, env={**os.environ, 'PYTHONPATH': 'src'})
if _st.returncode != 0:
    raise RuntimeError('elastic_augment self-test failed -- refusing to train')

# ---- train ---------------------------------------------------------------
_out_method = os.environ.get('BIOHUB_TRAIN_METHOD', 'unet_transformer_claude_elastic')
_cmd = [
    '/usr/bin/python3', 'scripts/train_unet_transformer.py',
    '--data-dir', str(TRAIN_DIR),
    '--splits', 'claude_finetune_splits.json', '--split', '0',
    '--method', _out_method,
    '--unet-weights', str(REPO_DIR / WEIGHTS_RELATIVE),
    '--epochs', os.environ.get('BIOHUB_TRAIN_EPOCHS', '30'),
    '--lr', os.environ.get('BIOHUB_TRAIN_LR', '3e-5'),
    # 2, not the pack's 8. Their 8 died on a P100 in the first forward pass with
    # `CUDA error: invalid configuration argument` -- a kernel launch whose grid exceeds
    # what sm_60 accepts, not an out-of-memory. Inference on the same card runs at
    # --unet-batch-size 4 and training holds activations for the backward pass on top, so
    # 2 is the conservative read of the one data point we have.
    '--batch-size', os.environ.get('BIOHUB_TRAIN_BATCH', '2'),
    '--max-iters', os.environ.get('BIOHUB_TRAIN_MAX_ITERS', '300'),
    '--num-workers', os.environ.get('BIOHUB_TRAIN_WORKERS', '2'),
    # Capacity. The defaults are the pretrained shapes, and they are the defaults for a
    # reason: change either one and the checkpoint's tensors stop matching, the full restore
    # refuses (BIOHUB_TRAIN_ALLOW_SCRATCH overrides), and the run becomes a from-scratch
    # training that has to beat a 400-epoch model inside what is left of the quota.
    #
    # BIOHUB_TRAIN_DOWNSAMPLE is the one capacity knob that costs nothing: convolutions do
    # not care about spatial extent, so '1,2,2' keeps every pretrained weight and gives the
    # detector 4x the resolution in Y and X. notes/04 measured detection as essentially the
    # whole contest. It costs ~4x the compute and memory, not parameters.
    '--unet-out-channels', os.environ.get('BIOHUB_TRAIN_OUT_CH', '32'),
    '--unet-layers', os.environ.get('BIOHUB_TRAIN_LAYERS', '32,64,128'),
    '--downsample', os.environ.get('BIOHUB_TRAIN_DOWNSAMPLE', '1,4,4'),
    '--window-size', '2', '--pool-kernel-um', '5.0',
    '--single-gpu',
]
print(' '.join(_cmd), flush=True)
_rc = subprocess.run(_cmd, cwd=REPO_DIR, env={**os.environ, 'PYTHONPATH': 'src'})
print('training rc =', _rc.returncode, flush=True)
if _rc.returncode != 0:
    raise RuntimeError(f'training failed rc={_rc.returncode}')

# ---- collect --------------------------------------------------------------
_weights_out = Path('/kaggle/working/claude_finetuned')
_weights_out.mkdir(parents=True, exist_ok=True)
_src_dir = Path('/kaggle/working/claude_weights') / _out_method / 'split_0'
for _f in sorted(_src_dir.iterdir()):
    shutil.copy2(_f, _weights_out / _f.name)
    print(f'  saved {_f.name} ({_f.stat().st_size:,} bytes)', flush=True)
(_weights_out / 'claude_finetune_manifest.json').write_text(json.dumps({
    'base_weights_sha256': _primary_actual_sha256,
    'holdout': _val, 'n_train': len(_tr),
    'augmentations': ['brightness_augment', 'flip_augment', 'elastic_augment'],
    'command': _cmd[1:],
}, indent=2))
print('DONE -- fine-tuned weights are in /kaggle/working/claude_finetuned', flush=True)
